# Sawti 6.3 — self-hosted serving benchmark (free Kaggle T4×2)

Runs the **service's own pipeline** (`sawti.agent.graph`: extraction prompt, schema,
grounding, scoring — from the uploaded repo snapshot, not reimplemented) against
Qwen3-8B served by **vLLM 0.18.1** on Kaggle's free 2× T4, with and without the
Phase 5 QLoRA adapter, plus a small Hugging Face transformers baseline. Scoring
happens later, locally, from the results file this notebook writes.

**Estimated GPU time: ~2 h (hard budget 2 h 50 min — no new calls start after it).**

| Stage | Estimate |
|---|---|
| Install vLLM, download Qwen3-8B (~16 GB), start the server (TP=2, eager) | ~25 min |
| Arm 1 — vLLM, adapter on, concurrency 1, 15 calls (5 per language) | ~30 min (~2 min/call at ~15 tok/s single-stream, ~1.7k output tokens) |
| Arm 2 — vLLM, adapter on, concurrency 16, 45 calls | ~10 min |
| Arm 3 — vLLM, adapter **off** (base Qwen3-8B), concurrency 16, 45 calls | ~10 min |
| Arm 4 — HF transformers fp16, adapter on, concurrency 1, 3 calls (1 per language) | ~25 min (load ~5 min + ~6 min/call) |
| Assemble results | ~1 min |

The per-call figures are estimates from the T4's memory bandwidth and from Phase 5's
HF timings, not measurements — measuring them is what this notebook is for.

### Before you run

1. **Settings → Accelerator: GPU T4 ×2. Internet: On.**
2. **Add data** (the three private datasets built by `scripts/build_kaggle_bundles.py`):
   `sawti-qlora-adapter`, `sawti-comparison-calls`, `sawti-repo`.
3. **Add-ons → Secrets → `HF_TOKEN`** (a Hugging Face read token), and attach it to this
   notebook. It is read with `kaggle_secrets` and never printed or written to disk.
4. Run all cells top to bottom (or **Save Version → Save & Run All** to run headless).

**If a session dies part-way:** download `sawti_runs.jsonl` from the output, upload it as a
dataset, attach it, and re-run — cell 3 picks it up and every arm skips the calls already
recorded. At most the calls in flight are lost.

**Output to download:** `/kaggle/working/sawti_kaggle_results.json` (plus
`vllm_server.log`, `feasibility.json`).

**Data:** the transcripts are synthetic and redacted. Note that sending them here *is*
sending them to a third party (Kaggle, i.e. Google); the zero-egress claim for
self-hosting applies to running the same server on your own hardware, not to this run.

In [ ]:
# ===== Cell 1 — FEASIBILITY CHECK. Everything below runs only if this passes. =====
# Installs vLLM in its own venv, starts it with --tensor-parallel-size 2 --dtype float16
# and the LoRA adapter, sends ONE real call (structured output, thinking off) and prints it.
import glob, json, os, signal, subprocess, sys, time, urllib.request
from pathlib import Path

START = time.time()
DEADLINE = START + 2 * 3600 + 50 * 60          # GPU-time budget: no new calls after this
VLLM_VERSION = "0.18.1"                        # docs/09-DECISIONS.md, 2026-10-07
BASE_MODEL = "Qwen/Qwen3-8B"
LORA_NAME = "sawti-qlora"
PORT = 8000
WORK = Path("/kaggle/working"); WORK.mkdir(exist_ok=True)
TMP = Path("/tmp/sawti"); TMP.mkdir(exist_ok=True)

def find(marker):
    """A dataset's root, found by a file only it contains (Kaggle's mount layout varies)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{marker}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"no attached dataset contains {marker} — add it via 'Add data'")
    return Path(hits[0]).parent

ADAPTER_DIR = find("adapter_model.safetensors")
CALLS_DIR = find("call_set.json")
REPO_DIR = find("BUNDLE.json")

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")   # never printed
os.environ["HF_HOME"] = str(TMP / "hf")

def sh(*cmd, **kw):
    print("$", " ".join(map(str, cmd))[:300]); return subprocess.run(list(map(str, cmd)), check=True, **kw)

gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total,driver_version",
                       "--format=csv,noheader"], capture_output=True, text=True).stdout.strip().splitlines()
print("GPUs:", gpus)
assert len(gpus) == 2 and all("T4" in g and "7.5" in g for g in gpus), "needs Accelerator: GPU T4 x2"

t = time.time()
sh(sys.executable, "-m", "pip", "install", "-q", "uv")
VLLM_ENV = TMP / "vllm-venv"
sh("uv", "venv", "-q", "--python", "3.12", VLLM_ENV)
sh("uv", "pip", "install", "-q", "--python", VLLM_ENV / "bin/python", f"vllm=={VLLM_VERSION}", "--torch-backend=auto")
install_s = time.time() - t

SERVER_ARGS = [
    BASE_MODEL, "--served-model-name", BASE_MODEL,
    "--tensor-parallel-size", "2", "--dtype", "float16",
    "--enable-lora", "--lora-modules", f"{LORA_NAME}={ADAPTER_DIR}", "--max-lora-rank", "16",
    "--max-model-len", "8192", "--max-num-seqs", "16", "--gpu-memory-utilization", "0.90",
    # The validated Kaggle T4x2 profile for vLLM 0.18.x: no FlashAttention on sm_75
    # (vLLM picks TRITON_ATTN), eager mode, no custom all-reduce over PCIe.
    "--enforce-eager", "--disable-custom-all-reduce",
    "--port", str(PORT),
]
log = open(WORK / "vllm_server.log", "w")
t = time.time()
server = subprocess.Popen([VLLM_ENV / "bin/vllm", "serve", *SERVER_ARGS], stdout=log, stderr=subprocess.STDOUT,
                          start_new_session=True, env=os.environ.copy())

def get(path, timeout=10):
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}{path}", timeout=timeout) as r:
        return r.read().decode()

while True:
    if server.poll() is not None:
        print(open(WORK / "vllm_server.log").read()[-6000:])
        raise RuntimeError("vLLM exited during start-up — see the log above (FEASIBILITY FAILED)")
    try:
        get("/health"); break
    except Exception:
        if time.time() - t > 40 * 60:
            raise TimeoutError("vLLM not healthy after 40 min (FEASIBILITY FAILED)")
        time.sleep(10)
startup_s = time.time() - t
models = [m["id"] for m in json.loads(get("/v1/models"))["data"]]
print("served models:", models)
assert LORA_NAME in models and BASE_MODEL in models, "adapter not served"

call_set = json.loads((CALLS_DIR / "call_set.json").read_text())
probe_id = call_set["ar"][0]
probe_text = (CALLS_DIR / "transcripts" / f"{probe_id}.txt").read_text()
schema = {"type": "object", "properties": {"summary": {"type": "string"}, "commitments": {"type": "integer"}},
          "required": ["summary", "commitments"]}
body = {"model": LORA_NAME, "temperature": 0.0, "max_tokens": 512,
        "messages": [{"role": "system", "content": "Summarize the call and count the agent's commitments."},
                     {"role": "user", "content": probe_text}],
        "response_format": {"type": "json_schema", "json_schema": {"name": "probe", "schema": schema}},
        "chat_template_kwargs": {"enable_thinking": False}}
req = urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/chat/completions", data=json.dumps(body).encode(),
                             headers={"Content-Type": "application/json"})
t = time.time()
with urllib.request.urlopen(req, timeout=900) as r:
    reply = json.loads(r.read())
probe_s = time.time() - t
content = reply["choices"][0]["message"]["content"]
print("probe call:", probe_id, f"{probe_s:.1f}s", reply["usage"])
print(content)
parsed = json.loads(content)                      # structured output really is JSON
assert "<think>" not in content, "thinking mode is on"
backend = [l.strip() for l in open(WORK / "vllm_server.log") if "attention backend" in l.lower() or "TRITON_ATTN" in l][:3]
FEASIBLE = True
json.dump({"ok": True, "vllm_version": VLLM_VERSION, "gpus": gpus, "server_args": [str(a) for a in SERVER_ARGS],
           "attention_backend_log": backend, "install_s": round(install_s), "startup_s": round(startup_s),
           "probe": {"call_id": probe_id, "latency_s": round(probe_s, 1), "usage": reply["usage"], "output": parsed}},
          open(WORK / "feasibility.json", "w"), indent=2)
print(f"\nFEASIBILITY PASSED — install {install_s/60:.1f} min, start-up {startup_s/60:.1f} min")

## Client environment — the uploaded repo snapshot, exact pins

In [ ]:
# ===== Cell 2 — sawti client in its own venv (9 packages pinned from uv.lock) =====
assert FEASIBLE, "cell 1 did not pass"
CLIENT_ENV = TMP / "client-venv"
sh("uv", "venv", "-q", "--python", "3.12", CLIENT_ENV)
CLIENT_PY = CLIENT_ENV / "bin/python"
sh("uv", "pip", "install", "-q", "--python", CLIENT_PY, "-r", REPO_DIR / "requirements-client.txt")
CLIENT_ENVVARS = {**os.environ, "PYTHONPATH": str(REPO_DIR / "src"),
                  "LANGFUSE_PUBLIC_KEY": "", "LANGFUSE_SECRET_KEY": ""}   # no tracing on Kaggle
sh(CLIENT_PY, "-c", "import sawti.eval.kaggle_benchmark, sawti.agent.graph; print('sawti client OK')",
   env=CLIENT_ENVVARS)
print(json.loads((REPO_DIR / "BUNDLE.json").read_text())["git_commit"], "(repo snapshot)")

In [ ]:
# ===== Cell 3 — resume: pick up a previous run's checkpoint if one is attached =====
CHECKPOINT = WORK / "sawti_runs.jsonl"
previous = sorted(glob.glob("/kaggle/input/**/sawti_runs.jsonl", recursive=True))
if previous and not CHECKPOINT.exists():
    CHECKPOINT.write_text(Path(previous[0]).read_text())
    print("resuming from", previous[0], "—", sum(1 for _ in open(CHECKPOINT)), "lines")
else:
    print("fresh run" if not CHECKPOINT.exists() else f"continuing {CHECKPOINT}")

def arm(engine, name, *args):
    """One arm through sawti.eval.kaggle_benchmark (the service pipeline), within the time budget."""
    sh(CLIENT_PY, "-m", "sawti.eval.kaggle_benchmark", engine, "--arm", name,
       "--calls-dir", CALLS_DIR, "--checkpoint", CHECKPOINT, "--deadline-epoch", str(DEADLINE), *args,
       env=CLIENT_ENVVARS)
    print(f"elapsed {(time.time() - START)/60:.0f} min of {(DEADLINE - START)/60:.0f}")

## vLLM arms

In [ ]:
# ===== Cell 4 — Arm 1: vLLM, adapter ON, concurrency 1 (latency), 15 calls =====
arm("vllm", "vllm_adapter_c1", "--model", LORA_NAME, "--concurrency", "1", "--limit", "15")

In [ ]:
# ===== Cell 5 — Arm 2: vLLM, adapter ON, concurrency 16 (throughput + accuracy), 45 calls =====
arm("vllm", "vllm_adapter_c16", "--model", LORA_NAME, "--concurrency", "16")

In [ ]:
# ===== Cell 6 — Arm 3: vLLM, adapter OFF (base Qwen3-8B), concurrency 16, 45 calls =====
# Same server, same engine: the difference to arm 2 is the adapter alone.
arm("vllm", "vllm_base_c16", "--model", BASE_MODEL, "--concurrency", "16")

In [ ]:
# ===== Cell 7 — stop vLLM and free both GPUs =====
os.killpg(os.getpgid(server.pid), signal.SIGTERM)   # the API server spawns an engine child: kill the group
try:
    server.wait(timeout=120)
except subprocess.TimeoutExpired:
    os.killpg(os.getpgid(server.pid), signal.SIGKILL)
for _ in range(60):
    used = [int(x) for x in subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                           capture_output=True, text=True).stdout.split()]
    if max(used) < 1000:
        break
    time.sleep(5)
print("GPU memory used (MiB):", used)

## HF transformers baseline — what justifies vLLM
fp16 across both T4s (`device_map="auto"`), the same precision as vLLM, adapter on, thinking off,
`sawti.llm.local_hf_provider.LocalHFProvider` with its own JSON retries (no guided decoding).

In [ ]:
# ===== Cell 8 — Arm 4: HF transformers, adapter ON, concurrency 1, 3 calls =====
sh("uv", "pip", "install", "-q", "--python", CLIENT_PY, "torch", "--torch-backend=auto")
sh("uv", "pip", "install", "-q", "--python", CLIENT_PY, "-r", REPO_DIR / "requirements-hf.txt")
if time.time() < DEADLINE - 20 * 60:
    arm("hf", "hf_adapter_c1", "--adapter", ADAPTER_DIR, "--limit", "3")
else:
    print("skipped: not enough of the GPU-time budget left for the HF baseline")

## Results — download `sawti_kaggle_results.json`

In [ ]:
# ===== Cell 9 — one results file =====
OUT = WORK / "sawti_kaggle_results.json"
sh(CLIENT_PY, "-m", "sawti.eval.kaggle_benchmark", "assemble", "--checkpoint", CHECKPOINT, "--out", OUT,
   "--meta", WORK / "feasibility.json", env=CLIENT_ENVVARS)
print(f"total {(time.time() - START)/60:.0f} min — download {OUT}")